# D173 — Window Function Exercises with Olist

This notebook contains 10 hands-on Olist exercises using MySQL window functions. Each problem includes requirements and an expected output shape, followed by an empty answer cell.

## Learning goals

- number and rank rows with `ROW_NUMBER`, `RANK`, and `DENSE_RANK`;
- compare adjacent rows with `LAG` and `LEAD`;
- identify partition boundaries with `FIRST_VALUE` and `LAST_VALUE`;
- calculate running totals and moving averages;
- control calculations with explicit `ROWS` window frames; and
- combine aggregation, CTEs, and window functions without changing row granularity unexpectedly.

These exercises target MySQL 8 or newer.


## 1. Connect to MySQL

The defaults match the earlier D16 and D17 notebooks. Environment variables can override them.


In [ ]:
import os
import mysql.connector
from mysql.connector import Error

connection = mysql.connector.connect(
    host=os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    port=int(os.environ.get("MYSQL_PORT", "3306")),
    user=os.environ.get("MYSQL_USERNAME", "root"),
    password=os.environ.get("MYSQL_PASSWORD", "root"),
    database=os.environ.get("MYSQL_DATABASE", "olist_import_lab"),
)

print("Connected:", connection.is_connected())
print("MySQL version:", connection.server_info)


## 2. Query helpers

`execute_sql` sends SQL to MySQL and prints up to 25 returned rows.


In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None, max_rows=25):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if not cursor.with_rows:
            connection.commit()
            print(f"Statement completed. Affected rows: {cursor.rowcount:,}")
            return cursor.rowcount

        columns = [item[0] for item in cursor.description]
        rows = cursor.fetchmany(max_rows + 1)
        visible_rows = rows[:max_rows]
        print_rows(columns, visible_rows)
        if len(rows) > max_rows:
            print(f"... showing the first {max_rows} rows")
        return visible_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


## 3. Olist tables used

| Table | Relevant columns |
|---|---|
| `olist_orders` | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` |
| `olist_order_items` | `order_id`, `order_item_id`, `product_id`, `seller_id`, `price`, `freight_value` |
| `olist_order_payments` | `order_id`, `payment_sequential`, `payment_type`, `payment_value` |
| `olist_customers` | `customer_id`, `customer_unique_id`, `customer_state` |
| `olist_products` | `product_id`, `product_category_name` |
| `product_category_translation` | `product_category_name`, `product_category_name_english` |
| `olist_order_reviews` | `order_id`, `review_score` |
| `olist_sellers` | `seller_id`, `seller_state` |

Remember that an order may contain several items and several payment rows. Aggregate each source to the required business level before joining when row multiplication could inflate a measure.


In [ ]:
execute_sql("""
SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'payments', COUNT(*) FROM olist_order_payments
UNION ALL SELECT 'customers', COUNT(*) FROM olist_customers
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
""")


## 4. Window function reminder

```sql
function_name(expression) OVER (
    PARTITION BY grouping_column
    ORDER BY ordering_column
    ROWS BETWEEN frame_start AND frame_end
)
```

- `PARTITION BY` restarts the calculation for each group.
- Window `ORDER BY` defines sequence inside each partition.
- A window frame selects rows relative to the current row.
- Window functions preserve detail rows; `GROUP BY` combines them.
- MySQL generally does not allow a window-function result in the same query block's `WHERE` clause. Calculate it in a CTE, then filter in the outer query.

### Important `LAST_VALUE` detail

With an ordered window, the default frame often ends at the current row. To return the actual last row in the entire partition, specify:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
```


## Exercise 1: Rank sellers by revenue within each state

Regional leaders want their strongest sellers ranked within the seller’s registered state.

**Task:** First aggregate `olist_order_items` to one row per seller with total item revenue. Join seller location, then calculate `ROW_NUMBER`, `RANK`, and `DENSE_RANK` within each `seller_state`, ordered by revenue descending. Use seller ID as a stable tie-breaker only for `ROW_NUMBER`.

**Must use:** aggregate CTE, `PARTITION BY`, `ROW_NUMBER`, `RANK`, and `DENSE_RANK`.

**Expected result:** Columns `seller_state`, `seller_id`, `seller_revenue`, `row_number_in_state`, `revenue_rank`, and `dense_revenue_rank`. Sort by state and row number. Observe how the three functions would treat equal revenue values.


## Exercise 2: Previous order value for each customer

Customer analytics wants to compare every buyer’s order payment with their previous order payment.

**Task:** Aggregate payment rows to one payment total per order. Join orders and customers, then use `LAG` within each `customer_unique_id`, ordered by purchase timestamp and order ID. Calculate the change from the previous order.

**Must use:** payment aggregation CTE, `LAG`, `PARTITION BY`, and deterministic window ordering.

**Expected result:** Columns `customer_unique_id`, `order_id`, `order_purchase_timestamp`, `order_payment`, `previous_order_payment`, and `change_from_previous`. The first order for every customer must have null previous value and null change.


## Exercise 3: Next purchase date and days until next order

Retention analysts want to know how long repeat buyers waited before ordering again.

**Task:** Join customers and orders. Use `LEAD` to retrieve each customer’s next purchase timestamp, ordered by timestamp and order ID. Calculate the number of days until that purchase using `DATEDIFF`.

**Must use:** `LEAD`, `PARTITION BY`, deterministic ordering, and `DATEDIFF`.

**Expected result:** Columns `customer_unique_id`, `order_id`, `order_purchase_timestamp`, `next_purchase_timestamp`, and `days_until_next_order`. The last order in each customer partition must have null next-date information.


## Exercise 4: First and last purchase for every customer order

The lifecycle team wants every order labeled with the buyer’s first and most recent purchase timestamps.

**Task:** Join customers and orders. For every order row, calculate `FIRST_VALUE` and `LAST_VALUE` of purchase timestamp within the customer partition. Use a full-partition frame for both expressions.

**Must use:** `FIRST_VALUE`, `LAST_VALUE`, and `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

**Expected result:** Columns `customer_unique_id`, `order_id`, `order_purchase_timestamp`, `first_purchase_timestamp`, and `last_purchase_timestamp`. First and last timestamps must remain constant across all rows belonging to the same buyer.


## Exercise 5: Seller monthly running revenue

The sales team wants cumulative seller revenue over time.

**Task:** Aggregate item revenue to one row per seller and purchase month. On that monthly result, calculate cumulative revenue within each seller from their first month through the current month.

**Must use:** monthly aggregate CTE, windowed `SUM`, `PARTITION BY`, and `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.

**Expected result:** Columns `seller_id`, `purchase_month`, `monthly_revenue`, and `running_revenue`; ordered by seller and month. Running revenue must never decrease for a seller.


## Exercise 6: Three-month moving average by seller

Sales operations wants a smoother view of monthly seller revenue.

**Task:** Aggregate item revenue by seller and purchase month. Calculate the average revenue over the current row and two preceding available monthly rows.

**Must use:** monthly aggregate CTE, windowed `AVG`, and `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`.

**Expected result:** Columns `seller_id`, `purchase_month`, `monthly_revenue`, and `three_row_moving_average`. Round the average to two decimals. Explain through the output that this is based on three available rows, which may not be three consecutive calendar months when a seller has an inactive month.


## Exercise 7: Centered review-score window

Customer experience wants a local review trend around each reviewed order for each seller.

**Task:** Join reviews, orders, and order items, but first ensure each seller/order pair appears once. For each seller, order reviewed orders by review creation date and order ID. Calculate the average review score across the previous row, current row, and next row.

**Must use:** deduplication or grouping CTE, windowed `AVG`, and `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING`.

**Expected result:** Columns `seller_id`, `order_id`, `review_creation_date`, `review_score`, and `centered_three_review_average`. At partition boundaries the average uses only available rows.


## Exercise 8: First and last product price within an order

Order-quality analysts want to compare every item with the first and last listed items in its order.

**Task:** For orders containing multiple items, retain item-level rows and use `order_item_id` as the sequence. Return each item price along with the first and last item price in that order. Make the frame explicit so `LAST_VALUE` reaches the end of the order.

**Must use:** a CTE to identify multi-item orders, `FIRST_VALUE`, `LAST_VALUE`, `PARTITION BY order_id`, and a full-partition `ROWS` frame.

**Expected result:** Columns `order_id`, `order_item_id`, `price`, `first_item_price`, and `last_item_price`. The boundary prices remain constant within an order, while the current `price` varies by item.


## Exercise 9: Month-over-month category revenue change

Merchandising wants to see whether category revenue increased or decreased from the previous active month.

**Task:** Join items, orders, products, and category translation. Aggregate to English category and purchase month, excluding null English names. Use `LAG` to retrieve the previous monthly revenue for that category, then calculate the absolute and percentage change. Protect percentage division with `NULLIF`.

**Must use:** aggregate CTE, `LAG`, category partitioning, `NULLIF`, and deterministic month ordering.

**Expected result:** Columns `category`, `purchase_month`, `monthly_revenue`, `previous_month_revenue`, `revenue_change`, and `revenue_change_percent`. The first row per category has null comparison values. Round money and percentage outputs to two decimals.


## Exercise 10: Top three products per category with comparison

The product team wants the three highest-revenue products in every English category and their gap from the category leader.

**Task:** Aggregate item revenue to one row per product and English category. In another CTE, use `DENSE_RANK` to rank products within category and `FIRST_VALUE` to capture the leading product revenue. In the outer query, keep ranks 1 through 3 and calculate the revenue gap from the leader.

**Must use:** chained CTEs, `DENSE_RANK`, `FIRST_VALUE`, `PARTITION BY`, an explicit full-partition frame, and outer filtering.

**Expected result:** Columns `category`, `product_id`, `product_revenue`, `revenue_rank`, `leader_revenue`, and `gap_from_leader`; sorted by category, rank, and product ID. Every rank-1 product has a zero gap.


## Review checklist

- Confirm the required business grain before applying a window function.
- Use `PARTITION BY` only when the calculation must restart by group.
- Add a stable tie-breaker to window ordering when dates or values can tie.
- Use an explicit `ROWS` frame for running, moving, first, and last calculations.
- Remember that `LAST_VALUE` needs an ending boundary of `UNBOUNDED FOLLOWING` to see the partition’s actual final row.
- Calculate window values in a CTE before filtering by rank or window result.
- Avoid joining multiple one-to-many detail tables before aggregation.
- Round only displayed results, not intermediate values used for calculations.


## Close the connection

Run this cell after completing the exercises.


In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")
